# pAIr 추론 서버 총괄 노트북 (RunPod)

**Pod 를 켤 때마다 이 노트북을 열고 `Run > Run All Cells`(또는 위에서부터 `Shift+Enter`)만 하면 됩니다.**
코드 받기/갱신 → 라이브러리·모델 준비 → 서버 실행 → 확인 → 백엔드에 넣을 값 출력까지 한 번에 합니다.

- **처음 한 번만** 서버 API 키·콜백 정보를 물어봅니다. 입력한 값은 영구 볼륨(`/workspace/.pair-vision.env`, 본인만 읽기)에 저장되어 **다음부터는 묻지 않고 같은 값**을 씁니다(백엔드 설정을 매번 바꿀 필요 없음).
- 라이브러리·모델은 영구 볼륨에 저장되어 **두 번째부터는 건너뜁니다**(Pod 를 만들 때 **Volume 을 반드시 붙이세요**. HTTP 포트 **8000** 공개, GPU 1장).
- 1~5번이 서버를 띄우는 부분이고, 6번(사진 재생 시험)과 7번(서버 끄기)은 필요할 때만 실행합니다. Run All 은 1~5번까지만 돌고 6·7번은 건너뜁니다(`RUN_TEST=True`로 바꾸면 6번도 실행).
- 칸에서 오류가 나면 **출력 전체**를 알려 주세요. 끝나면 **Pod 를 Stop/Terminate** 하세요(켜 두면 과금).

## 1. 설정

In [ ]:
import getpass
import json
import os
import secrets
import shutil
import subprocess
import sys
import time
import urllib.request as ur
from pathlib import Path

REPO_URL = "https://github.com/KosAmigo/pAIr-vision.git"   # 비공개면 처음 한 번 토큰을 물어본다
REPO_BRANCH = "main"
RUN_TEST = False          # True 로 바꾸면 6번(사진 재생 시험)도 Run All 에 포함

WORK = Path("/workspace") if Path("/workspace").exists() else Path.home()
ROOT = WORK / "pAIr-vision"
MODELS = ROOT / "models"
VENV_PY = str(WORK / "venv-vision" / "bin" / "python")
ENV_FILE = WORK / ".pair-vision.env"

# 저장된 설정(키·비밀·콜백 주소)을 읽는다. 없으면 처음 한 번만 물어보고 저장한다.
CFG = {}
if ENV_FILE.exists():
    CFG = json.loads(ENV_FILE.read_text())
    print("저장된 설정 사용:", ENV_FILE, "(바꾸려면 이 파일을 지우고 다시 실행)")
else:
    CFG["api_key"] = getpass.getpass("서버 API 키(비우면 자동 생성): ") or secrets.token_urlsafe(24)
    CFG["callback_url"] = input("백엔드 콜백 주소 https://<백엔드>/internal/vision/events (없으면 Enter): ").strip()
    CFG["callback_secret"] = ""
    if CFG["callback_url"]:
        CFG["callback_secret"] = (getpass.getpass("콜백 서명 비밀(백엔드 PAIR_VISION_EVENT_HMAC_SECRET 과 같은 값, 비우면 자동 생성): ")
                                  or secrets.token_urlsafe(24))
    ENV_FILE.write_text(json.dumps(CFG))
    ENV_FILE.chmod(0o600)
    print("설정 저장:", ENV_FILE)
API_KEY, CALLBACK_URL, CALLBACK_SECRET = CFG["api_key"], CFG["callback_url"], CFG["callback_secret"]

if shutil.which("nvidia-smi"):
    print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                 capture_output=True, text=True).stdout.strip())
else:
    print("nvidia-smi 없음 -> GPU 가 안 잡힌 Pod 일 수 있음(Pod 설정에서 GPU 확인)")
print("작업 폴더:", ROOT, "| python", sys.version.split()[0])

## 2. 코드 받기 / 갱신

In [ ]:
GIT_ENV = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if (ROOT / ".git").exists():
    r = subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], capture_output=True, text=True, env=GIT_ENV)
    print("코드 갱신:", (r.stdout + r.stderr).strip()[-200:])
else:
    url = REPO_URL
    if subprocess.run(["git", "ls-remote", url, REPO_BRANCH], capture_output=True, env=GIT_ENV).returncode != 0:
        token = getpass.getpass("저장소가 비공개입니다. GitHub 토큰(읽기 전용)을 붙여넣으세요: ")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    r = subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", url, str(ROOT)],
                       capture_output=True, text=True, env=GIT_ENV)
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", REPO_URL], check=False)  # 토큰 제거
    print(r.stdout[-300:], r.stderr[-300:].replace(url, "<url>"))
    assert r.returncode == 0, "git clone 실패"
os.chdir(ROOT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

## 3. 환경 준비 (라이브러리 · 모델)
`tools/runpod/bootstrap.sh` 가 **없는 것만** 준비합니다: 시스템 라이브러리(apt, Pod 를 켤 때마다 필요, 수 초) · 파이썬 환경(`/workspace/venv-vision`) · 모델 5개(약 700MB, sha256 검증).
처음 한 번만 오래 걸리고(수 분~십수 분) 다음부터는 몇 초입니다.

In [ ]:
boot = subprocess.Popen(["bash", "tools/runpod/bootstrap.sh"], cwd=ROOT, env={**os.environ, "WORK": str(WORK)},
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in boot.stdout:
    print(line, end="")
assert boot.wait() == 0, "환경 준비 실패 - 위 출력의 마지막 오류를 알려 주세요"

## 4. 서버 실행
이미 떠 있는 서버가 있으면 끄고 새로 띄웁니다(코드를 갱신했을 때 반영되도록). 모델 적재에 수십 초 걸립니다.

In [ ]:
subprocess.run(["pkill", "-f", "pair_vision"], check=False)
time.sleep(2)

env = {**os.environ, "VISION_API_KEY": API_KEY, "VISION_DETECTOR": "real", "VISION_MODELS_DIR": str(MODELS),
       "PYTHONPATH": str(ROOT / "src")}
if CALLBACK_URL:
    env["VISION_CALLBACK_URL"] = CALLBACK_URL
    env["VISION_CALLBACK_SECRET"] = CALLBACK_SECRET
LOG = WORK / "vision-server.log"
server = subprocess.Popen([VENV_PY, "-m", "pair_vision"], cwd=ROOT, env=env, stdout=open(LOG, "w"),
                          stderr=subprocess.STDOUT, start_new_session=True)  # 노트북을 닫아도 서버는 계속 돈다


def readyz():
    req = ur.Request("http://localhost:8000/readyz", headers={"Authorization": f"Bearer {API_KEY}"})
    return json.load(ur.urlopen(req, timeout=3))


ready = None
for _ in range(90):
    time.sleep(3)
    if server.poll() is not None:
        break
    try:
        ready = readyz()
        break
    except Exception:
        continue
if ready:
    print(ready)
    print("\n>>> modelsLoaded =", ready["modelsLoaded"], "| gpu =", ready["gpu"],
          "" if ready["gpu"] else "  <- False 면 GPU 를 못 쓰는 중(위 1번의 GPU 줄과 Pod 설정 확인)")
else:
    print("서버가 안 떴습니다. 로그 마지막 줄:\n" + "".join(LOG.read_text().splitlines(keepends=True)[-20:]))

## 5. 외부 접속 확인 · 백엔드에 넣을 값
RunPod 프록시 주소로 `/healthz` 와 **WebSocket(`/v1/stream`)** 이 통과하는지 봅니다(Pod 에서 HTTP 포트 8000 을 공개해 둬야 함).
아래 값을 **백엔드 환경 변수**에 넣습니다. 비밀이 출력되니 옮긴 뒤 노트북 출력을 지우세요(Edit > Clear Outputs).
**백엔드의 실제 추론 서버 클라이언트는 아직 구현 전(가짜만 있음)이라, 지금은 서버 단독 동작까지만 확인됩니다.**

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID", "")
BASE_URL = f"https://{pod_id}-8000.proxy.runpod.net" if pod_id else ""
print("서버 주소(추정):", BASE_URL or "RUNPOD_POD_ID 가 없습니다 - Pod 화면의 Connect > HTTP Service 주소를 쓰세요")
if BASE_URL:
    r = subprocess.run([VENV_PY, "tools/runpod/check_proxy.py", BASE_URL, API_KEY], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout.strip(), r.stderr[-300:])
    if r.returncode != 0:
        print("-> /healthz 나 WebSocket 이 안 되면 포트 공개 여부, 프록시의 WebSocket 차단 여부를 확인")
print()
print("PAIR_VISION_INFERENCE_URL     =", BASE_URL or "<Pod 화면의 8000 포트 주소>")
print("PAIR_VISION_INFERENCE_KEY     =", API_KEY)
print("PAIR_VISION_EVENT_HMAC_SECRET =", CALLBACK_SECRET or "(콜백 주소를 안 넣어서 없음)", "  (서버 콜백 키 번호 k1)")
print("PAIR_SERVER_VISION_ENABLED    = false  (테스트 회차에서만 켠다)")

## 6. 사진 재생 시험 (필요할 때만)
`RUN_TEST = True` 로 바꾸거나 이 칸만 따로 실행하세요. 사진은 **이름순**으로 8초씩 서버에 보내집니다.
- `/workspace/photos/phone/` : 휴대폰 캠 사진(책상 옆 시점, 키보드·마우스·두 손) -> 손 이탈, 태블릿·메모·스마트워치 이벤트
- `/workspace/photos/webcam/` : 웹캠 사진. **이름순 첫 사진은 정면**이어야 합니다(앞 3초로 기준 자세). 그다음 고개를 돌린 사진 등
폴더가 비어 있으면 합성 이미지로 속도만 잽니다. 응시자 실사진은 올리지 마세요(동의한 사내 인원 사진만).

In [ ]:
if not RUN_TEST:
    print("RUN_TEST=False 라 건너뜀 (이 칸만 실행하려면 위 1번에서 RUN_TEST=True 로 바꾸고 이 칸을 실행)")
else:
    IMG = (".png", ".jpg", ".jpeg")
    HOLD = 8

    def count_images(folder):
        return len([p for p in folder.glob("*") if p.suffix.lower() in IMG]) if folder.is_dir() else 0

    def replay(channel, folder, max_edge, quality):
        n = count_images(folder)
        duration = max(24, n * HOLD + 4)
        cmd = [VENV_PY, "tools/replay/replay.py", "--api-key", API_KEY, "--photos", str(folder), "--channel", channel,
               "--hold", str(HOLD), "--duration", str(duration), "--settle", "6", "--max-edge", str(max_edge),
               "--jpeg-quality", str(quality)]
        print(f"== {channel} 채널: 사진 {n}장, {duration}초 ==")
        r = subprocess.run(cmd, cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
        print(r.stdout[-2500:], r.stderr[-800:])

    phone_dir, webcam_dir = WORK / "photos" / "phone", WORK / "photos" / "webcam"
    if count_images(phone_dir) == 0 and count_images(webcam_dir) == 0:
        synthetic = WORK / "synthetic"
        synthetic.mkdir(exist_ok=True)
        subprocess.run([VENV_PY, "-c", (
            "from PIL import Image, ImageDraw; img = Image.new('RGB', (512, 384), (200, 200, 200)); "
            "ImageDraw.Draw(img).rectangle((120, 200, 380, 330), fill=(60, 60, 60)); img.save(r'%s')" % (synthetic / "synthetic.jpg"))],
            check=True)
        print("사진이 없어 합성 이미지 사용(속도만 의미 있음)")
        phone_dir = synthetic
    if count_images(phone_dir):
        replay("phone", phone_dir, 512, 85)       # 계약: 휴대폰 maxEdge 512
    if count_images(webcam_dir):
        replay("webcam", webcam_dir, 400, 55)     # 계약: 웹캠 maxEdge 400, 품질 0.55

    req = ur.Request("http://localhost:8000/metrics", headers={"Authorization": f"Bearer {API_KEY}"})
    for line in ur.urlopen(req, timeout=5).read().decode().splitlines():
        if line.startswith(("pair_vision_frame_latency_seconds_bucket", "pair_vision_frames_", "pair_vision_events_total",
                            "pair_vision_detector_errors_total")) and not line.startswith("#"):
            print(line)
    print("\n서버 로그의 '프레임 분석 실패':", sum("프레임 분석 실패" in l for l in LOG.read_text().splitlines()), "건")

## 7. 서버 끄기 (끝났을 때만 직접 실행)
**Run All 이 이 칸까지 실행해도 서버가 꺼지지 않도록, 아래 칸은 `STOP = True` 로 바꿨을 때만 동작합니다.**
끈 뒤에는 **RunPod 화면에서 Pod 를 Stop/Terminate** 하세요.

In [ ]:
STOP = False
if STOP:
    subprocess.run(["pkill", "-f", "pair_vision"], check=False)
    print("서버 종료")
else:
    print("STOP=False 라 서버를 그대로 둡니다 (끄려면 STOP=True 로 바꾸고 이 칸 실행)")